# Load PDF into vector store

In [ ]:
from rich import print

In [ ]:
from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader

loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True,
    use_multithreading=True
)

In [ ]:
documents=loader.load()

In [ ]:
# print(documents)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n","\n"," ",""]
)

In [ ]:
chunks=text_splitter.split_documents(documents=documents)

In [ ]:
# print(chunks)

In [ ]:
print(len(documents))
print(len(chunks))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
vectorstore.save_local("../data/faiss_index")

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [ ]:
query = "Backend skills of Riyaz?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}")
    print("-" * 50)
    print(doc.page_content[:500])
    print("\nMetadata:", doc.metadata)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
groq_api_key=os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm=ChatGroq(groq_api_key=groq_api_key,model="meta-llama/llama-4-scout-17b-16e-instruct")

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_core.prompts import ChatPromptTemplate


system_prompt = """
You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I could not find the answer in the provided documents."

Keep answers concise and accurate.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# ---------------------------------------------------
# Document Chain
# ---------------------------------------------------

question_answer_chain = create_stuff_documents_chain(
    llm,
    prompt
)

# ---------------------------------------------------
# RAG Chain
# ---------------------------------------------------

rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

print("RAG chain ready")

In [ ]:
response = rag_chain.invoke({
    "input": "Backend skills of Riyaz?"
})

print(response["answer"])

In [ ]:
response = rag_chain.invoke({
    "input": "Backend skills of Riyaz?"
})

print("\nANSWER:\n")
print(response["answer"])

print("\nRETRIEVED DOCUMENTS:\n")

for i, doc in enumerate(response["context"], 1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print(doc.page_content[:300])
    print(doc.metadata)